In [1]:
import torch
from icecream import ic

In [2]:
extent = torch.tensor([[0.0, 3.0], [0.0, 4.0]]).to("mps")
histogram_shape = (3, 4)
positions = torch.rand(100, 100_000, 2).to("mps")

In [3]:
%%timeit
bin_widths = (extent[..., 1] - extent[..., 0]) / torch.tensor(
    histogram_shape, device=positions.device
)

The slowest run took 5.51 times longer than the fastest. This could mean that an intermediate result is being cached.
6.39 ms ± 3.74 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [4]:
del extent, histogram_shape, positions

In [5]:
bins = (
    torch.linspace(0.0, 1.0, 500).to("mps"),
    torch.linspace(0.0, 1.0, 500).to("mps"),
)

In [6]:
%%timeit
# Grid dimensions and spacing validation
grid_sizes = []
spacings = []
for i, bin_array in enumerate(bins):
    N = bin_array.numel()
    if N < 2:
        raise ValueError(f"bins[{i}] must have at least 2 elements")

    spacing = bin_array[1] - bin_array[0]
    if N > 2:
        diffs = torch.diff(bin_array)
        if not torch.allclose(diffs, spacing, rtol=1e-4):
            raise ValueError(f"bins[{i}] must have uniform spacing")

    grid_sizes.append(N)
    spacings.append(spacing)

15.9 ms ± 701 μs per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [7]:
del bins

In [8]:
extent = torch.tensor([[0.0, 3.0], [0.0, 4.0]]).to("mps")
positions = torch.rand(100, 100_000, 2).to("mps")
charges = torch.ones_like(positions[..., 0])

In [9]:
%%timeit
# Set weights to zero for particles outside grid bounds
inside_mask = ((extent[..., 0] <= positions) & (positions < extent[..., 1])).all(
    dim=-1
)  # Shape (..., num_samples)
masked_charges = charges * inside_mask  # Shape (..., num_samples)

32.7 ms ± 12.9 ms per loop (mean ± std. dev. of 7 runs, 100 loops each)


In [10]:
del extent, positions, charges

In [11]:
positions = (torch.rand(100, 100_000).to("mps"), torch.rand(100, 100_000).to("mps"))
bins = (
    torch.linspace(0.0, 1.0, 500).to("mps"),
    torch.linspace(0.0, 1.0, 500).to("mps"),
)
weights = torch.ones_like(positions[0]).to("mps")

In [12]:
%%timeit
# Set weights to zero for particles outside grid bounds
for pos, bin_array in zip(positions, bins):
    outside_mask = (pos < bin_array[0]) | (pos >= bin_array[-1])
    weights_dummy = weights * (~outside_mask).float()

24.3 ms ± 10.6 ms per loop (mean ± std. dev. of 7 runs, 100 loops each)


In [13]:
del positions, bins, weights

In [14]:
extent = torch.tensor([[0.0, 3.0], [0.0, 4.0]]).to("mps")
histogram_shape = (3, 4)
positions = torch.rand(100, 100_000, 2).to("mps")
bin_widths = (extent[..., 1] - extent[..., 0]) / torch.tensor(
    histogram_shape, device=positions.device
)

In [15]:
%%timeit
# Normalise particle coordinates to normalised bin space
positions_in_bin_space = (positions - extent[..., 0]) / bin_widths
positions_in_bin_space_int_component = positions_in_bin_space.floor().long()
positions_in_bin_space_fractional_components = (
    positions_in_bin_space - positions_in_bin_space_int_component
)

97.1 ms ± 5.69 ms per loop (mean ± std. dev. of 7 runs, 100 loops each)


In [16]:
del extent, histogram_shape, positions, bin_widths

In [17]:
positions = (torch.rand(100, 100_000).to("mps"), torch.rand(100, 100_000).to("mps"))
bins = (
    torch.linspace(0.0, 1.0, 500).to("mps"),
    torch.linspace(0.0, 1.0, 500).to("mps"),
)
grid_sizes = []
spacings = []
for i, bin_array in enumerate(bins):
    N = bin_array.numel()
    if N < 2:
        raise ValueError(f"bins[{i}] must have at least 2 elements")

    spacing = bin_array[1] - bin_array[0]
    if N > 2:
        diffs = torch.diff(bin_array)
        if not torch.allclose(diffs, spacing, rtol=1e-4):
            raise ValueError(f"bins[{i}] must have uniform spacing")

    grid_sizes.append(N)
    spacings.append(spacing)

In [18]:
%%timeit
# Normalize particle coordinates to grid index space
grid_indices = []
fractional_parts = []
for pos, bin_array, spacing in zip(positions, bins, spacings):
    # Normalized coordinate in grid index space
    u = (pos - bin_array[0]) / spacing

    # Left cell index
    i = torch.floor(u).to(torch.int64)
    grid_indices.append(i)

    # Fractional distance to right cell
    w = u - i
    fractional_parts.append(w)

34.7 ms ± 3.11 ms per loop (mean ± std. dev. of 7 runs, 100 loops each)


In [19]:
del positions, bins, grid_sizes, spacings, i, bin_array, spacing, diffs

In [20]:
num_hist_dims = 2
extent = torch.tensor([[0.0, 3.0], [0.0, 4.0]]).to("mps")
histogram_shape = (3, 4)
positions = torch.rand(100, 100_000, 2).to("mps")
charges = torch.ones_like(positions[..., 0])
bin_widths = (extent[..., 1] - extent[..., 0]) / torch.tensor(
    histogram_shape, device=positions.device
)
inside_mask = ((extent[..., 0] <= positions) & (positions < extent[..., 1])).all(
    dim=-1
)  # Shape (..., num_samples)
masked_charges = charges * inside_mask  # Shape (..., num_samples)
positions_in_bin_space = (positions - extent[..., 0]) / bin_widths
positions_in_bin_space_int_component = positions_in_bin_space.floor().long()
positions_in_bin_space_fractional_components = (
    positions_in_bin_space - positions_in_bin_space_int_component
)

In [21]:
%%timeit
# Generate all corner combinations and their weights
num_corners = 2**num_hist_dims
# TODO speed: torch.tensor(list(itertools.product([0, 1], repeat=D)), device=device)
corner_offsets = (
    torch.arange(num_corners).unsqueeze(-1).to(positions.device)
    // (2 ** torch.arange(num_hist_dims).to(positions.device))
    % 2
)
corner_positions_in_bin_space = (
    positions_in_bin_space_int_component.unsqueeze(-2) + corner_offsets
)
clamped_corner_positions_in_bin_space = corner_positions_in_bin_space.clamp(
    corner_positions_in_bin_space.new_zeros(()),
    corner_positions_in_bin_space.new_tensor(histogram_shape) - 1,
)
corner_weight_factors = positions_in_bin_space_fractional_components.unsqueeze(
    -2
).where(
    corner_offsets == 1,
    1.0 - positions_in_bin_space_fractional_components.unsqueeze(-2),
)  # Shape (..., num_samples, num_corners, num_hist_dims)
corner_weights = corner_weight_factors.prod(
    dim=-1
)  # Shape (..., num_samples, num_corners)
corner_charges = (  # Actual charge deposition on the corners
    masked_charges.unsqueeze(-1) * corner_weights
)  # Shape (..., num_samples, num_corners)

323 ms ± 138 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [22]:
del (
    num_hist_dims,
    extent,
    histogram_shape,
    positions,
    charges,
    bin_widths,
    inside_mask,
    masked_charges,
    positions_in_bin_space,
    positions_in_bin_space_int_component,
    positions_in_bin_space_fractional_components,
)

In [23]:
num_dims = 2
positions = (torch.rand(100, 100_000).to("mps"), torch.rand(100, 100_000).to("mps"))
weights = torch.ones_like(positions[0]).to("mps")
bins = (
    torch.linspace(0.0, 1.0, 500).to("mps"),
    torch.linspace(0.0, 1.0, 500).to("mps"),
)
grid_sizes = []
spacings = []
for i, bin_array in enumerate(bins):
    N = bin_array.numel()
    if N < 2:
        raise ValueError(f"bins[{i}] must have at least 2 elements")

    spacing = bin_array[1] - bin_array[0]
    if N > 2:
        diffs = torch.diff(bin_array)
        if not torch.allclose(diffs, spacing, rtol=1e-4):
            raise ValueError(f"bins[{i}] must have uniform spacing")

    grid_sizes.append(N)
    spacings.append(spacing)
# Normalize particle coordinates to grid index space
grid_indices = []
fractional_parts = []
for pos, bin_array, spacing in zip(positions, bins, spacings):
    # Normalized coordinate in grid index space
    u = (pos - bin_array[0]) / spacing

    # Left cell index
    i = torch.floor(u).to(torch.int64)
    grid_indices.append(i)

    # Fractional distance to right cell
    w = u - i
    fractional_parts.append(w)

In [24]:
%%timeit
# Generate all corner combinations and their weights
num_corners = 2**num_dims
corner_indices = []
corner_weights = []

for corner in range(num_corners):
    # Determine which corners to use based on binary representation
    corner_offsets = []
    weight_factors = []

    for dim in range(num_dims):
        if (corner >> dim) & 1:  # Use right cell in this dimension
            corner_offsets.append(1)
            weight_factors.append(fractional_parts[dim])
        else:  # Use left cell in this dimension
            corner_offsets.append(0)
            weight_factors.append(1 - fractional_parts[dim])

    # Calculate indices for this corner
    corner_idx_list = []
    for dim in range(num_dims):
        base_idx = grid_indices[dim]
        offset_idx = (base_idx + corner_offsets[dim]).clamp(0, grid_sizes[dim] - 1)
        corner_idx_list.append(offset_idx)

    # Calculate weight for this corner
    corner_weight = weights
    for weight_factor in weight_factors:
        corner_weight = corner_weight * weight_factor

    corner_indices.append(corner_idx_list)
    corner_weights.append(corner_weight)

90.7 μs ± 1.55 μs per loop (mean ± std. dev. of 7 runs, 10,000 loops each)


In [25]:
del (
    num_dims,
    positions,
    weights,
    bins,
    grid_sizes,
    spacings,
    i,
    bin_array,
    spacing,
    diffs,
    grid_indices,
    fractional_parts,
    pos,
    # bin_array,
    # spacing,
    u,
    # i,
    w,
)

In [ ]:
positions = torch.rand(100, 100_000, 2).to("mps")
histogram_shape = (500, 500)
num_hist_dims = 2
clamped_corner_positions_in_bin_space = torch.rand(100, 100_000, 4, num_hist_dims).to(
    "mps"
)
corner_charges = torch.rand(100, 100_000, 4).to("mps")

In [ ]:
%%timeit
vector_shape = positions.shape[:-2]
charge_grid = corner_charges.new_zeros(*vector_shape, *histogram_shape)

vector_indices = tuple(
    torch.arange(this_vector_dim_length).reshape(
        this_vector_dim_length, *([1] * (num_hist_dims - dim_idx + 1))
    )
    for dim_idx, this_vector_dim_length in enumerate(vector_shape)
)

charge_grid = charge_grid.index_put(
    vector_indices + clamped_corner_positions_in_bin_space.unbind(-1),
    corner_charges,
    accumulate=True,
)

In [ ]:
del (
    positions,
    histogram_shape,
    num_hist_dims,
    clamped_corner_positions_in_bin_space,
    corner_charges,
)